# Super-Wised Coding LLM 1.0 — Kaggle Free GPU Training & Smoke Test

This notebook trains and verifies the complete **from-scratch Batch 1 coding LLM** (JavaScript, TypeScript, Node.js, Express.js, and NestJS) on a **Kaggle free GPU (T4 / P100)** or CPU:
- Directly clones and executes code from [https://github.com/munisekhar605/super-wised-code-llm-1.0](https://github.com/munisekhar605/super-wised-code-llm-1.0).
- Streams and trains on real open-source JavaScript code from the public, ungated Hugging Face dataset [Nan-Do/code-search-net-javascript](https://huggingface.co/datasets/Nan-Do/code-search-net-javascript) (138,000+ real functions).
- Custom decoder-only Transformer (`src/transformer/`) with **RoPE**, **RMSNorm**, **SwiGLU**, **Grouped-Query Attention (GQA)**, and **causal self-attention** initialized with random weights.
- Uses the `bigcode/starcoder2-15b` **tokenizer only** (`src/tokenizer/`; no pretrained weights are loaded).
- End-to-end verification: real data streaming, secret filtering, deduplication, tokenization, binary dataset validation, training, automatic checkpoint resume, interactive testing, and SafeTensors release packaging.

> **Dataset Note:** `Nan-Do/code-search-net-javascript` is **fully public and ungated**. It does not require any authorization or gated token approval.

In [ ]:
import glob
import os
import pathlib
import shutil
import subprocess
import zipfile
import torch

REPO_URL = "https://github.com/munisekhar605/super-wised-code-llm-1.0.git"

def setup_kaggle_workspace() -> pathlib.Path:
    cwd = pathlib.Path.cwd()
    if (cwd / 'pyproject.toml').exists() and (cwd / 'src' / 'transformer' / 'model.py').exists():
        return cwd
    if (cwd.parent / 'pyproject.toml').exists() and (cwd.parent / 'src' / 'transformer' / 'model.py').exists():
        os.chdir(cwd.parent)
        return cwd.parent

    working_root = pathlib.Path('/kaggle/working/super-wised-code-llm-1.0')
    if (working_root / 'pyproject.toml').exists() and (working_root / 'src' / 'transformer' / 'model.py').exists():
        os.chdir(working_root)
        return working_root

    # Primary: Clone repository directly from GitHub
    try:
        print(f"Cloning repository from {REPO_URL} into {working_root}...")
        subprocess.check_call(['git', 'clone', REPO_URL, str(working_root)])
        if (working_root / 'pyproject.toml').exists():
            os.chdir(working_root)
            return working_root
    except Exception as exc:
        print(f"Git clone failed or unavailable ({exc}). Checking Kaggle input datasets...")

    for marker in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True):
        src_dir = pathlib.Path(marker).parent
        if (src_dir / 'src' / 'transformer' / 'model.py').exists():
            shutil.copytree(src_dir, working_root, dirs_exist_ok=True)
            os.chdir(working_root)
            return working_root

    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    if zips:
        extract_root = pathlib.Path('/kaggle/working')
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(extract_root)
        for marker in glob.glob('/kaggle/working/**/pyproject.toml', recursive=True):
            found = pathlib.Path(marker).parent
            os.chdir(found)
            return found

    raise FileNotFoundError(
        f"Unable to set up workspace. Could not clone {REPO_URL} and no /kaggle/input dataset found."
    )

project_dir = setup_kaggle_workspace()
print('Working directory:', project_dir)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device:', torch.cuda.get_device_name(0))

## 1. Verify Dependencies

In [ ]:
import importlib.util
import subprocess
import sys

required_modules = ['torch', 'transformers', 'tokenizers', 'huggingface_hub', 'datasets', 'numpy', 'safetensors']
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    print('Installing missing modules:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
else:
    print('All required modules are installed.')

## 2. Run Unit Test Suite

In [ ]:
!python -m unittest discover -s tests -v

## 3. Prepare and Validate Real Batch 1 Dataset from Nan-Do/code-search-net-javascript

Streams real open-source JavaScript code from `Nan-Do/code-search-net-javascript` (ungated, public), screens for secrets, deduplicates, and packs into contiguous uint32 binary token files (`data/tokenized/train.bin`, `data/tokenized/val.bin`).

In [ ]:
# Stream real JavaScript code from public Hugging Face dataset Nan-Do/code-search-net-javascript
!python scripts/prepare_data.py \
    --hf-dataset Nan-Do/code-search-net-javascript \
    --text-column code \
    --allow-unlicensed-after-review \
    --output-dir data/tokenized \
    --tokenizer bigcode/starcoder2-15b \
    --max-records 10000 \
    --min-split-tokens 300

!python scripts/prepare_data.py --validate-only --output-dir data/tokenized --min-split-tokens 300

## 4. Train Small Configurable Model and Verify Automatic Checkpoint Resume

In [ ]:
!python scripts/train.py --config configs/model.json --training-config configs/training.json --data-dir data/tokenized --output-dir checkpoints --log-file logs/metrics.jsonl --no-resume --max-steps 30

In [ ]:
!python scripts/train.py --config configs/model.json --training-config configs/training.json --data-dir data/tokenized --output-dir checkpoints --log-file logs/metrics.jsonl --resume --max-steps 50

## 5. Evaluate Validation Perplexity and Batch 1 Code Generation

## 6. Test Model Code Generation Interactively

Test completions generated by the model on Batch 1 JavaScript, TypeScript, and Express.js prompts.

In [ ]:
import argparse
from src.inference.generate import run_inference

test_prompts = [
    "// JavaScript function to calculate total with tax\nexport function calculateTotal(items, taxRate) {\n",
    "// Express.js GET route handler for /api/health\napp.get('/api/health', (req, res) => {\n",
    "// TypeScript memory cache class\nexport class CacheService<T> {\n"
]

print('=== Interactive Code Generation Test ===')
for prompt in test_prompts:
    print('\n' + '=' * 60)
    print('PROMPT:\n' + prompt.strip())
    completion = run_inference(argparse.Namespace(
        checkpoint='checkpoints/best.pt',
        tokenizer='bigcode/starcoder2-15b',
        prompt=prompt,
        max_new_tokens=48,
        temperature=0.7,
        top_k=40
    ))
    print('-' * 40)
    print('GENERATED OUTPUT:\n' + completion.strip())

## 7. Package and Publish Release Artifacts

Exports `model.safetensors`, `model_config.json`, `training_state.json`, tokenizer files, and model card into `/kaggle/working/super-wised-code-llm-1.0/release`. Set `HF_TOKEN` and add `--push` to upload to the Hugging Face Hub.

In [ ]:
# Package model into release directory
!python scripts/publish.py \
    --checkpoint checkpoints/best.pt \
    --config configs/model.json \
    --repo-id YOUR_HF_USERNAME/super-wised-code-llm-1.0 \
    --output-dir release

print('Listing packaged release files in release/:')
!ls -lh release

In [ ]:
# (Optional) Upload release package directly to Hugging Face Hub
# To upload, uncomment below and ensure your Hugging Face write token is set:
# os.environ["HF_TOKEN"] = "hf_YOUR_WRITE_TOKEN"
# !python scripts/publish.py --checkpoint checkpoints/best.pt --config configs/model.json --repo-id YOUR_HF_USERNAME/super-wised-code-llm-1.0 --output-dir release --push